In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from tqdm import tqdm

# -------------------------------
#  Neural Network with BatchNorm
# -------------------------------
class BSDENet(nn.Module):
    """
    Neural network that maps X (d-dimensional) to Z (d-dimensional).
    Architecture: two hidden layers (d+10 units each) with ReLU and BatchNorm.
    Weight initialization can be 'normal', 'uniform', or 'default' (Kaiming uniform).
    """
    def __init__(self, d, init_type='normal'):
        super().__init__()
        self.net = nn.Sequential(
            nn.BatchNorm1d(d),
            nn.Linear(d, d+10),
            nn.BatchNorm1d(d+10),
            nn.ReLU(),
            nn.Linear(d+10, d+10),
            nn.BatchNorm1d(d+10),
            nn.ReLU(),
            nn.Linear(d+10, d)
        )
        self._init_weights(init_type)

    def _init_weights(self, init_type):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                if init_type == 'normal':
                    # Normal distribution with mean 0, std 0.1
                    nn.init.normal_(m.weight, mean=0.0, std=0.1)
                    if m.bias is not None:
                        nn.init.constant_(m.bias, 0)
                elif init_type == 'uniform':
                    # Uniform distribution in [-0.1, 0.1]
                    nn.init.uniform_(m.weight, a=-0.1, b=0.1)
                    if m.bias is not None:
                        nn.init.constant_(m.bias, 0)
                # If 'default', do nothing (PyTorch's default initialization)
            elif isinstance(m, nn.BatchNorm1d):
                # BatchNorm defaults: weight=1, bias=0
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
                nn.init.normal_(m.bias, 0, 0.1)
                nn.init.uniform_(m.weight, 0.1, 0.5)

    def forward(self, x):
        return self.net(x)

# -------------------------------
#  Deep BSDE Solver Class
# -------------------------------
class DeepBSDESolver:
    def __init__(self, T, N, d, xi, mu, sigma, f, g,optimizer="Adam",
                 batch_size=64, lr=1e-3, device='cpu'):
        """
        T: terminal time
        N: number of time steps
        d: dimension
        xi: initial state (tensor of shape (d,))
        mu: drift function mu(t, x) -> (batch, d)
        sigma: diffusion function sigma(t, x) -> (batch, d, d) or (batch, d) if diagonal
        f: nonlinearity f(t, x, y, z) -> (batch,)
        g: terminal condition g(x) -> (batch,)
        batch_size: mini-batch size
        lr: learning rate
        device: 'cpu' or 'cuda'
        """
        self.T = T
        self.N = N
        self.d = d
        self.xi = xi.to(device)
        self.mu = mu
        self.sigma = sigma
        self.f = f
        self.g = g
        self.batch_size = batch_size
        self.device = device

        self.dt = T / N
        self.sqrt_dt = np.sqrt(self.dt)

        # Learnable parameters: initial value Y0 and initial gradient Z0
        self.Y0 = nn.Parameter(torch.randn(1, device=device) * 0.1)
        self.Z0 = nn.Parameter(torch.randn(d, device=device) * 0.1)

        # Create N-1 neural networks (one for each interior time step)
        self.nets = nn.ModuleList([BSDENet(d).to(device) for _ in range(N-1)])

        # Collect all parameters for optimizer
        all_params = list(self.nets.parameters()) + [self.Y0, self.Z0]
        if optimizer=="Adam":
             self.optimizer = optim.Adam(all_params, lr=lr)
        else :
           # optimizer="SGD"
           self.optimizer = optim.SGD(all_params, lr=lr)

        self.grad_clip_max_norm = 1.0  # for gradient clipping
        self.delta_clip=100
        self.lr_boundaries = [1000, 2000, 3000]  # à adapter selon l'exemple
        self.scheduler = torch.optim.lr_scheduler.MultiStepLR(
            self.optimizer, milestones=self.lr_boundaries, gamma=0.1
        )

    def _simulate_batch(self):
        """
        Simulate one batch of forward SDE paths and Brownian increments.
        Returns:
            X: tensor of shape (batch_size, N+1, d)  [X_0,...,X_N]"
            dW: tensor of shape (batch_size, N, d)   increments
        """
        batch_size = self.batch_size
        d = self.d
        device = self.device

        # Initialise X[0] for all samples
        X = torch.zeros(batch_size, self.N+1, d, device=device)
        X[:, 0, :] = self.xi.expand(batch_size, d)

        # Pre-allocate dW
        dW = torch.randn(batch_size, self.N, d, device=device) * self.sqrt_dt

        # Euler-Maruyama
        for n in range(self.N):
            t = n * self.dt
            Xn = X[:, n, :]
            mu_val = self.mu(t, Xn)
            sigma_val = self.sigma(t, Xn)

            # If sigma returns a matrix (batch, d, d), we need matrix multiplication.
            # For diagonal case, we can just elementwise multiply.
            # Here we assume diagonal (common in examples) for simplicity.
            # For full matrix, adjust accordingly.
            X[:, n+1, :] = Xn + mu_val * self.dt + sigma_val * dW[:, n, :]

        return X, dW

    def _compute_loss(self, X, dW):
        """
        Given simulated paths X and increments dW, compute the loss.
        """
        batch_size = X.shape[0]
        device = self.device

        # Initialise Y with the learnable Y0 (expanded to batch)
        Y = self.Y0.expand(batch_size)

        # Use learnable Z0 at n=0
        Z = self.Z0.expand(batch_size, self.d)

        # Iterate over time steps
        for n in range(self.N):
            t = n * self.dt
            Xn = X[:, n, :]          # (batch, d)
            dWn = dW[:, n, :]         # (batch, d)

            if n > 0:
                # For n>=1, use neural network to predict Z
                Z = self.nets[n-1](Xn)   # (batch, d)

            # Compute f(t, Xn, Y, Z)  (should return (batch,))
            fn = self.f(t, Xn, Y, Z)

            # Update Y
            Y = Y - fn * self.dt + torch.sum(Z * dWn, dim=1)
            Y = torch.clamp(Y, min=-1000000.0, max=1000000.0)

        # Final loss: MSE between Y_N and g(X_N)
        XN = X[:, -1, :]

        # loss = torch.mean((Y - self.g(XN))**2)
        delta = Y - self.g(XN)

        #perte de type Huber
        loss = torch.where(torch.abs(delta) < self.delta_clip,
                   delta**2,
                   2 * self.delta_clip * torch.abs(delta) - self.delta_clip**2)
        loss = torch.mean(loss)
        
        return loss

    def train(self, num_iterations, log_interval=100):
        """
        Run the optimisation loop.
        """
        self.nets.train()
        losses = []

        iterator = tqdm(range(num_iterations), desc="Training")
        for it in iterator:
            self.optimizer.zero_grad()

            # Simulate a fresh batch of paths
            X, dW = self._simulate_batch()

            # Compute loss
            loss = self._compute_loss(X, dW)

            # Check for NaN
            if torch.isnan(loss):
                print("NaN loss encountered, stopping training")
                break

            # Backward pass
            loss.backward()

             #Gradient clipping to prevent explosion
            params = list(self.nets.parameters()) + [self.Y0, self.Z0]
            torch.nn.utils.clip_grad_norm_(params, max_norm=self.grad_clip_max_norm)

            self.optimizer.step()
            self.scheduler.step()

            losses.append(loss.item())
            iterator.set_postfix(loss=loss.item())

            if (it+1) % log_interval == 0:
                print(f"Iter {it+1:5d} | Loss = {loss.item():.6f} | Y0 = {self.Y0.item():.6f}")

        return losses

    @torch.no_grad()
    def evaluate(self, num_samples=256):
        """
        Evaluate the trained solver on fresh samples.
        Returns:
            mean_Y0: average of Y0 (should be close to u(0,xi))
            std_Y0: standard deviation of Y0 across batches
            mean_loss: average loss
        """
        self.nets.eval()
        Y0_list = []
        loss_list = []
        for _ in range(num_samples // self.batch_size):
            X, dW = self._simulate_batch()
            loss = self._compute_loss(X, dW)
            loss_list.append(loss.item())
            Y0_list.append(self.Y0.item())
        return np.mean(Y0_list), np.std(Y0_list), np.mean(loss_list),np.std(loss_list)


# -------------------------------
#  Example: Allen–Cahn (100d)
# -------------------------------
def allen_cahn_example():
    # PDE parameters
    d = 100
    T = 0.3
    N = 20
    xi = torch.zeros(d)               # initial point

    # Drift and diffusion (mu=0, sigma = sqrt(2)*I)
    def mu(t, x):
        return torch.zeros_like(x)

    def sigma(t, x):
        # For diagonal diffusion, return (batch, d) scaling factors
        return torch.full_like(x, np.sqrt(2.0))

    # Nonlinearity f(y, z) = y - y^3 (independent of t,x)
    def f(t, x, y, z):
        return y - y**3

    # Terminal condition g(x) = 1/(2 + 0.4 * ||x||^2)
    def g(x):
        return 1.0 / (2.0 + 0.4 * torch.sum(x**2, dim=1))

    # Reference solution u(0,0) (from branching diffusion method)
    u0_ref = 0.052802

    # Instantiate solver
    solver = DeepBSDESolver(T, N, d, xi, mu, sigma, f, g,
                            batch_size=64, lr=5e-4, device='cuda' if torch.cuda.is_available() else 'cpu')

    print(f"Device: {solver.device}")
    print(f"Reference u(0,xi) = {u0_ref:.6f}")

    # Train
    losses = solver.train(num_iterations=1000, log_interval=500)

    # Evaluate
    mean_Y0, std_Y0, mean_loss, std_loss = solver.evaluate(num_samples=10000)
    rel_error = abs(mean_Y0 - u0_ref) / abs(u0_ref)
    print("\n=== Final Results ===")
    print(f"Estimated u(0,xi) = {mean_Y0:.6f} ± {std_Y0:.6f}")
    print(f"Relative L1 error  = {rel_error:.4f}")
    print(f"Final loss         = {mean_loss:.6f} ± {std_loss:.6f}")

if __name__ == "__main__":
    allen_cahn_example()

KeyboardInterrupt: 

In [ ]:

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from tqdm import tqdm
from dataclasses import dataclass
from typing import Tuple, Optional
import optuna

# -------------------------------
#  Configuration des hyperparamètres
# -------------------------------
@dataclass
class HPConfig:
    # Loss function
    loss_type: str = 'huber'          # 'mse' ou 'huber'
    huber_delta: float = 50.0

    # Optimizer
    optimizer: str = 'adam'            # 'sgd', 'momentum', 'rmsprop', 'adam'
    learning_rate: float = 5e-4
    momentum: float = 0.9               # pour SGD momentum et RMSprop
    rmsprop_alpha: float = 0.99         # pour RMSprop
    adam_betas: Tuple[float, float] = (0.9, 0.999)
    adam_epsilon: float = 1e-8

    # Initialisation
    init_y0: str = 'uniform'            # 'uniform' ou 'normal'
    init_z0: str = 'uniform'            # 'uniform' ou 'normal'
    init_y0_range: Tuple[float, float] = (0.1, 0.5)   # si uniforme
    init_z0_range: Tuple[float, float] = (-0.1, 0.1)  # si uniforme
    init_y0_std: float = 0.1            # si normal
    init_z0_std: float = 0.1            # si normal

    # Autres
    batch_size: int = 64
    grad_clip: Optional[float] = 1.0    # clipping de gradient (None pour désactiver)
    use_bias: bool = False              # utiliser des biais dans les couches linéaires

    # Scheduler
    use_scheduler: bool = True
    lr_milestones: Tuple[int, ...] = (1000, 2000, 3000)
    lr_gamma: float = 0.1

# -------------------------------
#  Neural Network avec BatchNorm (option sans biais)
# -------------------------------
class BSDENet(nn.Module):
    def __init__(self, d, use_bias=False):
        super().__init__()
        self.net = nn.Sequential(
            nn.BatchNorm1d(d),
            nn.Linear(d, d+10, bias=use_bias),
            nn.BatchNorm1d(d+10),
            nn.ReLU(),
            nn.Linear(d+10, d+10, bias=use_bias),
            nn.BatchNorm1d(d+10),
            nn.ReLU(),
            nn.Linear(d+10, d, bias=use_bias)
        )
        # Initialisation spécifique des BatchNorm (comme dans l'article)
        for m in self.modules():
            if isinstance(m, nn.BatchNorm1d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
                nn.init.normal_(m.bias, 0, 0.1)
                nn.init.uniform_(m.weight, 0.1, 0.5)

    def forward(self, x):
        return self.net(x)

# -------------------------------
#  Solveur Deep BSDE configurable
# -------------------------------
class DeepBSDESolver:
    def __init__(self, T, N, d, xi, mu, sigma, f, g, config: HPConfig, device='cpu'):
        self.config = config
        self.T = T
        self.N = N
        self.d = d
        self.xi = xi.to(device)
        self.mu = mu
        self.sigma = sigma
        self.f = f
        self.g = g
        self.device = device

        self.dt = T / N
        self.sqrt_dt = np.sqrt(self.dt)

        # Initialisation de Y0 et Z0
        if config.init_y0 == 'uniform':
            self.Y0 = nn.Parameter(torch.empty(1, device=device).uniform_(*config.init_y0_range))
        elif config.init_y0 == 'normal':
            self.Y0 = nn.Parameter(torch.randn(1, device=device) * config.init_y0_std)
        else:
            raise ValueError("init_y0 must be 'uniform' or 'normal'")

        if config.init_z0 == 'uniform':
            self.Z0 = nn.Parameter(torch.empty(d, device=device).uniform_(*config.init_z0_range))
        elif config.init_z0 == 'normal':
            self.Z0 = nn.Parameter(torch.randn(d, device=device) * config.init_z0_std)
        else:
            raise ValueError("init_z0 must be 'uniform' or 'normal'")

        # Création des réseaux
        self.nets = nn.ModuleList([BSDENet(d, use_bias=config.use_bias).to(device) for _ in range(N-1)])

        # Paramètres pour l'optimiseur
        all_params = list(self.nets.parameters()) + [self.Y0, self.Z0]

        # Choix de l'optimiseur
        if config.optimizer.lower() == 'sgd':
            self.optimizer = optim.SGD(all_params, lr=config.learning_rate)
        elif config.optimizer.lower() == 'momentum':
            self.optimizer = optim.SGD(all_params, lr=config.learning_rate, momentum=config.momentum)
        elif config.optimizer.lower() == 'rmsprop':
            self.optimizer = optim.RMSprop(all_params, lr=config.learning_rate,
                                           alpha=config.rmsprop_alpha, momentum=config.momentum)
        elif config.optimizer.lower() == 'adam':
            self.optimizer = optim.Adam(all_params, lr=config.learning_rate,
                                        betas=config.adam_betas, eps=config.adam_epsilon)
        else:
            raise ValueError(f"Optimizer {config.optimizer} not supported")

        # Scheduler
        if config.use_scheduler:
            self.scheduler = torch.optim.lr_scheduler.MultiStepLR(
                self.optimizer, milestones=list(config.lr_milestones), gamma=config.lr_gamma
            )
        else:
            self.scheduler = None

    def _simulate_batch(self):
        batch_size = self.config.batch_size
        d = self.d
        device = self.device

        X = torch.zeros(batch_size, self.N+1, d, device=device)
        X[:, 0, :] = self.xi.expand(batch_size, d)

        dW = torch.randn(batch_size, self.N, d, device=device) * self.sqrt_dt

        for n in range(self.N):
            t = n * self.dt
            Xn = X[:, n, :]
            mu_val = self.mu(t, Xn)
            sigma_val = self.sigma(t, Xn)
            X[:, n+1, :] = Xn + mu_val * self.dt + sigma_val * dW[:, n, :]

        return X, dW

    def _compute_loss(self, X, dW):
        batch_size = X.shape[0]
        Y = self.Y0.expand(batch_size)
        Z = self.Z0.expand(batch_size, self.d)

        for n in range(self.N):
            t = n * self.dt
            Xn = X[:, n, :]
            dWn = dW[:, n, :]

            if n > 0:
                Z = self.nets[n-1](Xn)

            fn = self.f(t, Xn, Y, Z)
            Y = Y - fn * self.dt + torch.sum(Z * dWn, dim=1)
            # Clipping de sécurité (large)
            Y = torch.clamp(Y, min=-1e6, max=1e6)

        XN = X[:, -1, :]
        delta = Y - self.g(XN)

        if self.config.loss_type == 'mse':
            loss = torch.mean(delta**2)
        elif self.config.loss_type == 'huber':
            delta_clip = self.config.huber_delta
            loss = torch.where(torch.abs(delta) < delta_clip,
                               delta**2,
                               2 * delta_clip * torch.abs(delta) - delta_clip**2)
            loss = torch.mean(loss)
        else:
            raise ValueError("loss_type must be 'mse' or 'huber'")
        return loss

    def train(self, num_iterations, log_interval=100):
        self.nets.train()
        losses = []

        iterator = tqdm(range(num_iterations), desc="Training")
        for it in iterator:
            self.optimizer.zero_grad()
            X, dW = self._simulate_batch()
            loss = self._compute_loss(X, dW)

            if torch.isnan(loss):
                print("NaN loss encountered, stopping training")
                break

            loss.backward()

            if self.config.grad_clip is not None:
                params = list(self.nets.parameters()) + [self.Y0, self.Z0]
                torch.nn.utils.clip_grad_norm_(params, max_norm=self.config.grad_clip)

            self.optimizer.step()
            if self.scheduler is not None:
                self.scheduler.step()

            losses.append(loss.item())
            iterator.set_postfix(loss=loss.item())

            if (it+1) % log_interval == 0:
                print(f"Iter {it+1:5d} | Loss = {loss.item():.6f} | Y0 = {self.Y0.item():.6f}")

        return losses

    @torch.no_grad()
    def evaluate(self, num_samples=256):
        self.nets.eval()
        Y0_list = []
        loss_list = []
        for _ in range(num_samples // self.config.batch_size):
            X, dW = self._simulate_batch()
            loss = self._compute_loss(X, dW)
            loss_list.append(loss.item())
            Y0_list.append(self.Y0.item())
        return np.mean(Y0_list), np.std(Y0_list), np.mean(loss_list), np.std(loss_list)


# -------------------------------
#  Exemple : Allen–Cahn (100d) avec Optuna
# -------------------------------
def allen_cahn_params():
    d = 100
    T = 0.3
    N = 20
    xi = torch.zeros(d)

    def mu(t, x):
        return torch.zeros_like(x)

    def sigma(t, x):
        return torch.full_like(x, np.sqrt(2.0))

    def f(t, x, y, z):
        return y - y**3

    def g(x):
        return 1.0 / (2.0 + 0.4 * torch.sum(x**2, dim=1))

    return T, N, d, xi, mu, sigma, f, g

def objective(trial):
    # Suggestions d'hyperparamètres
    config = HPConfig(
        loss_type=trial.suggest_categorical('loss_type', ['mse', 'huber']),
        huber_delta=trial.suggest_float('huber_delta', 10, 200, log=True),
        optimizer=trial.suggest_categorical('optimizer', ['adam', 'momentum', 'rmsprop']),
        learning_rate=trial.suggest_float('learning_rate', 1e-5, 1e-2, log=True),
        momentum=trial.suggest_float('momentum', 0.8, 0.99),
        batch_size=trial.suggest_categorical('batch_size', [64, 128, 256]),
        init_y0=trial.suggest_categorical('init_y0', ['uniform', 'normal']),
        init_z0=trial.suggest_categorical('init_z0', ['uniform', 'normal']),
        grad_clip=trial.suggest_float('grad_clip', 0.1, 10.0, log=True),
        use_scheduler=trial.suggest_categorical('use_scheduler', [True, False]),
        lr_gamma=trial.suggest_float('lr_gamma', 0.1, 0.5),
    )

    # Paramètres fixes pour Allen-Cahn
    T, N, d, xi, mu, sigma, f, g = allen_cahn_params()
    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    solver = DeepBSDESolver(T, N, d, xi, mu, sigma, f, g, config, device=device)
    solver.train(num_iterations=2000, log_interval=500)  # moins d'itérations pour l'optimisation
    _, _, mean_loss, _ = solver.evaluate(num_samples=1000)

    # On minimise la perte
    return mean_loss

if __name__ == "__main__":
    # Création de l'étude Optuna
    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=50, timeout=3600)  # 50 essais ou 1h

    print("Best trial:")
    trial = study.best_trial
    print(f"  Loss: {trial.value}")
    print("  Params: ")
    for key, value in trial.params.items():
        print(f"    {key}: {value}")

    # Entraînement final avec les meilleurs hyperparamètres
    best_config = HPConfig(**trial.params)
    T, N, d, xi, mu, sigma, f, g = allen_cahn_params()
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    final_solver = DeepBSDESolver(T, N, d, xi, mu, sigma, f, g, best_config, device=device)
    final_solver.train(num_iterations=4000, log_interval=500)
    mean_y0, std_y0, mean_loss, std_loss = final_solver.evaluate(num_samples=10000)
    print(f"\nFinal result: Y0 = {mean_y0:.6f} ± {std_y0:.6f}, loss = {mean_loss:.6f} ± {std_loss:.6f}")

c:\Users\djana\OneDrive\Documents\MASEF\general_venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2026-03-17 21:36:09,054] A new study created in memory with name: no-name-6bcb673c-4e0b-4621-945b-fc1e88cf724c
Training:  25%|██▌       | 501/2000 [00:36<01:32, 16.15it/s, loss=0.00195]

Iter   500 | Loss = 0.001517 | Y0 = 0.052787


Training:  50%|█████     | 1001/2000 [01:12<01:23, 11.92it/s, loss=4.09e-5]

Iter  1000 | Loss = 0.000048 | Y0 = 0.053268


Training:  75%|███████▌  | 1500/2000 [01:54<01:02,  8.00it/s, loss=3.46e-5]

...

Training:  75%|███████▌  | 1501/2000 [03:42<01:14,  6.74it/s, loss=0.856]  

Iter  1500 | Loss = 0.584619 | Y0 = 0.297836


Training: 100%|██████████| 2000/2000 [04:30<00:00,  7.40it/s, loss=0.387]  
[I 2026-03-17 22:37:57,952] Trial 15 finished with value: 0.556612661906651 and parameters: {'loss_type': 'huber', 'huber_delta': 29.014133198245197, 'optimizer': 'adam', 'lr': 1.1285259795670165e-05, 'momentum': 0.9865191905663223, 'batch_size': 128, 'init_y0': 'uniform', 'init_z0': 'normal', 'grad_clip': 2.6369384296464675, 'use_scheduler': False, 'lr_gamma': 0.17090696940525682}. Best is trial 5 with value: 3.2575144966055326e-05.


Iter  2000 | Loss = 0.386971 | Y0 = 0.292267
Best trial:
  Loss: 3.2575144966055326e-05
  Params: 
    loss_type: mse
    huber_delta: 111.15860008551233
    optimizer: adam
    lr: 0.0021766784089924563
    momentum: 0.9200682564842129
    batch_size: 256
    init_y0: uniform
    init_z0: normal
    grad_clip: 0.19464785235140222
    use_scheduler: False
    lr_gamma: 0.29197601591581424


TypeError: HPConfig.__init__() got an unexpected keyword argument 'lr'

In [ ]:
class Fonction():
    def __init__():
        pass

    def allen_cahn_params(self):
        d = 100
        T = 0.3
        N = 20
        xi = torch.zeros(d)

        def mu(t, x):
            return torch.zeros_like(x)

        def sigma(t, x):
            return torch.full_like(x, np.sqrt(2.0))

        def f(t, x, y, z):
            return y - y**3

        def g(x):
            return 1.0 / (2.0 + 0.4 * torch.sum(x**2, dim=1))

        return T, N, d, xi, mu, sigma, f, g
    
    def hjb_equation_params(self):
        d=100
        N=20
        T=1
        xi = torch.zeros(d)

        def mu(t, x):
            return torch.zeros_like(x)

        def sigma(t, x):
            return torch.full_like(x, np.sqrt(2.0))

        def f(t, x, y, z):
            return -torch.sum(z**2, dim=1)

        def g(x):
            return torch.log(0.5 + 0.5 * torch.sum(x**2, dim=1))
        
        return T, N, d, xi, mu, sigma, f, g
    
    def hjb_equation_params(self):
        d=100
        N=20
        T=1
        xi = torch.zeros(d)

        def mu(t, x):
            return torch.zeros_like(x)

        def sigma(t, x):
            return torch.full_like(x, np.sqrt(2.0))

        def f(t, x, y, z):
            return -torch.sum(z**2, dim=1)

        def g(x):
            return torch.log(0.5 + 0.5 * torch.sum(x**2, dim=1))
        
        return T, N, d, xi, mu, sigma, f, g
    
    def EFD_params(self):
        mu_bar=6/100
        sigma_bar=2/10
        R_l=4/100
        R_b=6/100

        d=100
        N=20
        T=0.5
        xi = torch.ones(d)*100

        def mu(t, x):
            return x*mu_bar

        def sigma(t, x):
            return torch.full_like(x, np.sqrt(2.0))

        def f(t, x, y, z):
            return -R_l*y-(mu_bar-R_l)/sigma_bar + torch.sum(z, axis=1)+ (R_b-R_l)*torch.max(0, (1/sigma_bar)*torch.sum(z, axis=1)-y)

        def g(x):
            return torch.max(torch.max(x)-120,0)-2*torch.max(torch.max(x)-120,0)
        
        return T, N, d, xi, mu, sigma, f, g
    
    def MBT_PDE_params(self):
        d=100
        N=80
        T=1
        xi = torch.zeros(d)

        def mu(t, x):
            return torch.zeros_like(x)

        def sigma(t, x):
            return torch.full_like(x, d/np.sqrt(2.0))

        def f(t, x, y, z):
            return -(y-(2+d)/2*d)*torch.sum(z, axis=1)

        def g(x):
            tmp=torch.exp(T+torch.mean(x,axis=1))
            return tmp/(1+tmp)
        
        return T, N, d, xi, mu, sigma, f, g
    
    def QGD_PDE_params(self):
        d=100
        N=80
        T=1
        xi = torch.zeros(d)

        alpha=4/10

        def psi(t,x):
            return torch.sin((T-t+ (1/d)*torch.sum(x**2, dim=1))**alpha)

        def mu(t, x):
            return torch.zeros_like(x)

        def sigma(t, x):
            return torch.full_like(x, 1)

        def f(t, x, y, z):
            return # ajouter 

        def g(x):
            return torch.sin(((1/d)*torch.sum(x**2, dim=1))**(2*alpha))
            
        
        return T, N, d, xi, mu, sigma, f, g
    
    def TDRDT_PDE_params(self): 
        pass # Completer 
    



In [ ]:
import torch
import numpy as np

class Fonction:
    """
    Fournit les paramètres de différentes EDP/BSDE pour des tests numériques.
    Chaque méthode retourne un tuple (T, N, d, xi, mu, sigma, f, g) où :
        T : horizon temporel
        N : nombre de pas de temps
        d : dimension de l'espace
        xi : condition initiale (tenseur de taille d)
        mu : drift (fonction de t et x)
        sigma : diffusion (fonction de t et x)
        f : terme source (fonction de t, x, y, z)
        g : condition terminale (fonction de x)
    """

    def allen_cahn_params(self):
        """Équation d'Allen–Cahn (classique)."""
        d = 100
        T = 0.3
        N = 20
        xi = torch.zeros(d)

        def mu(t, x):
            return torch.zeros_like(x)

        def sigma(t, x):
            return torch.full_like(x, np.sqrt(2.0))

        def f(t, x, y, z):
            # z n'est pas utilisé ici
            return y - y**3

        def g(x):
            # x: (batch_size, d)
            return 1.0 / (2.0 + 0.4 * torch.sum(x**2, dim=1))

        return T, N, d, xi, mu, sigma, f, g

    def hjb_equation_params(self):
        """Équation de Hamilton–Jacobi–Bellman (HJB) classique."""
        d = 100
        N = 20
        T = 1.0
        xi = torch.zeros(d)

        def mu(t, x):
            return torch.zeros_like(x)

        def sigma(t, x):
            return torch.full_like(x, np.sqrt(2.0))

        def f(t, x, y, z):
            return -torch.sum(z**2, dim=1)   # z est de taille (batch, d)

        def g(x):
            return torch.log(0.5 + 0.5 * torch.sum(x**2, dim=1))

        return T, N, d, xi, mu, sigma, f, g

    def EFD_params(self):
        """
        Paramètres pour un problème de finance (Expected Future Dividends?).
        Adapté de l'article original.
        """
        mu_bar = 6 / 100
        sigma_bar = 2 / 10
        R_l = 4 / 100
        R_b = 6 / 100

        d = 100
        N = 20
        T = 0.5
        xi = torch.ones(d) * 100.0

        def mu(t, x):
            # x: (batch, d)
            return x * mu_bar   # drift proportionnel

        def sigma(t, x):
            return sigma_bar * x

        def f(t, x, y, z):
            # z: (batch, d)
            sum_z = torch.sum(z, dim=1)
            term1 = -R_l * y
            term2 = -(mu_bar - R_l) / sigma_bar
            term3 = sum_z
            term4 = (R_b - R_l) * torch.clamp((1.0 / sigma_bar) * sum_z - y, min=0)
            return term1 + term2 + term3 + term4

        def g(x):
            max_x = torch.max(x, dim=1)[0]          # max par échantillon
            penalty = torch.clamp(max_x - 120, min=0)
            return penalty - 2 *torch.clamp(max_x - 150, min=0)          # = -penalty (simplification)
            # Si l'intention était une différence, il faut revoir l'expression.

        return T, N, d, xi, mu, sigma, f, g

    def MBT_PDE_params(self):
        """
        Paramètres pour une EDP liée à un problème de type "Mountain Brook Test" ?
        """
        d = 20
        N = 80
        T = 1.0
        xi = torch.zeros(d)

        def mu(t, x):
            return torch.zeros_like(x)

        def sigma(t, x):
            return torch.full_like(x, d / np.sqrt(2.0))

        def f(t, x, y, z):
            # On suppose : f = -(y - (2+d)/(2*d)) * sum(z)
            coeff = (2.0 + d) / (2.0 * d)
            return (y - coeff) * torch.sum(z, dim=1)

        def g(x):
            tmp = torch.exp(T + torch.mean(x, dim=1))
            return tmp / (1 + tmp)

        return T, N, d, xi, mu, sigma, f, g

    def QGD_PDE_params(self):
        """
        Paramètres pour une EDP avec croissance quadratique (Quadratic Growth Diffusion).
        La fonction psi(t,x) est utilisée dans le terme source.
        """
        d = 100
        N = 80
        T = 1.0
        xi = torch.zeros(d)

        alpha = 4.0 / 10.0

        def psi(t, x):
            arg = (T - t) + (1.0 / d) * torch.sum(x**2, dim=1)
            return torch.sin(arg ** alpha)

        def mu(t, x):
            return torch.zeros_like(x)

        def sigma(t, x):
            return torch.full_like(x, 1.0)

        def f(t, x, y, z):
            # Forme plausible : f = -psi(t,x) * sum(z)   (ou autre combinaison)
            return -psi(t, x) * torch.sum(z, dim=1)

        def g(x):
            arg_term = (1.0 / d) * torch.sum(x**2, dim=1)
            return torch.sin(arg_term ** (2 * alpha))

        return T, N, d, xi, mu, sigma, f, g

    def TDRDT_PDE_params(self):
        """
        Paramètres pour une EDP de type "Time-Dependent Reaction-Diffusion-Transport".
        À compléter selon le problème spécifique.
        """
        d = 100
        N = 80
        T = 1.0
        xi = torch.zeros(d)

        # Exemple de coefficients (à adapter)
        def mu(t, x):
            return torch.zeros_like(x)

        def sigma(t, x):
            return torch.full_like(x, 1.0)

        def f(t, x, y, z):
            # Exemple : terme de réaction linéaire
            return -0.1 * y

        def g(x):
            # Exemple : condition terminale sinusoïdale
            return torch.sin(0.1 * torch.sum(x, dim=1))

        return T, N, d, xi, mu, sigma, f, g

In [ ]:
import torch
from torch import nn
from tqdm import tqdm

class NeuralNetwork(nn.Module):
    def __init__(self, d):
        super(NeuralNetwork, self).__init__()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(d, d + 10),
            nn.ReLU(),
            nn.Linear(d + 10, d + 10),
            nn.ReLU(),
            nn.Linear(d + 10, d),
        )

    def forward(self, x):
        return self.linear_relu_stack(x)

class ModelTrainer:
    def __init__(self, model, optimizer, learning_rate=1e-3):
        self.model = model
        self.loss_fn = nn.MSELoss()
        self.optimizer = torch.optim.Adam(self.model.parameters(), lr=learning_rate)
        
        # Historique des pertes
        self.history = {"train_loss": [], "test_loss": []}

    def train_step(self, dataloader):
        """Exécute une seule époque d'entraînement."""
        self.model.train() # Mode entraînement
        for X, y in dataloader:
            # Forward compute
            pred = self.model(X)
            loss = self.loss_fn(pred, y)

            # Backpropagation
            self.optimizer.zero_grad()
            loss.backward()
            self.optimizer.step()
            self.scheduler.step()

    def evaluate(self, X_train, Y_train, X_test, Y_test):
        """Calcule et enregistre les pertes actuelles."""
        self.model.eval() # Mode évaluation
        with torch.no_grad():
            train_loss = self.loss_fn(self.model(X_train), Y_train).item()
            test_loss = self.loss_fn(self.model(X_test), Y_test).item()
        
        self.history["train_loss"].append(train_loss)
        self.history["test_loss"].append(test_loss)
        return train_loss, test_loss

    def fit(self, train_dataloader, X_train, Y_train, X_test, Y_test, epochs=2000):
        """Boucle principale d'entraînement."""
        for epoch in tqdm(range(epochs)):
            # Calcul de la perte avant l'étape d'optimisation (optionnel)
            train_l, test_l = self.evaluate(X_train, Y_train, X_test, Y_test)
            
            if epoch % 200 == 0:
                print(f"Epoch {epoch} | Train Loss: {train_l:.6f} | Test Loss: {test_l:.6f}")
            
            # Entraînement sur le dataloader
            self.train_step(train_dataloader)